# Conv2d vs ConvTranspose2d

One picture to keep:

- **Conv2d gathers**: each output pixel = dot(kernel, a k×k patch of the input). Many inputs → one output.
- **ConvTranspose2d scatters**: each input pixel stamps `value × kernel` onto a k×k patch of the output, overlaps add. One input → many outputs.

Same kernel, opposite direction. That's why decoders (e.g. a conv decoder for sharp UWM frames) use it to go small → big.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_printoptions(precision=1, sci_mode=False)

## 1. Conv2d gathers: 4×4 → 2×2 (k=3, stride 1)

Kernel of all ones, so each output = sum of its 3×3 patch.


In [ ]:
x = torch.arange(16.0).reshape(1, 1, 4, 4)
conv = nn.Conv2d(1, 1, kernel_size=3, bias=False)
nn.init.ones_(conv.weight)
x, conv(x)

(tensor([[[[ 0.,  1.,  2.,  3.],
           [ 4.,  5.,  6.,  7.],
           [ 8.,  9., 10., 11.],
           [12., 13., 14., 15.]]]]),
 tensor([[[[45., 54.],
           [81., 90.]]]], grad_fn=<ConvolutionBackward0>))

Top-left output = 0+1+2+4+5+6+8+9+10 = 45. Four outputs, each read 9 inputs.


## 2. ConvTranspose2d scatters: 2×2 → 4×4 (same k=3, stride 1)

Input of all ones, kernel of all ones → each output cell shows **how many stamps landed on it**.


In [3]:
y = torch.ones(1, 1, 2, 2)
convT = nn.ConvTranspose2d(1, 1, kernel_size=3, bias=False)
nn.init.ones_(convT.weight)
convT(y)

tensor([[[[1., 2., 2., 1.],
          [2., 4., 4., 2.],
          [2., 4., 4., 2.],
          [1., 2., 2., 1.]]]], grad_fn=<ConvolutionBackward0>)

Centre cells got 4 stamps (all four inputs reach them), corners 1. Shape went back up: 2×2 → 4×4.


## 3. Scatter by hand

Loop over input pixels, add `value × kernel` into the output. Should match exactly.


In [ ]:
def conv_transpose_by_hand(y, w, stride=1):
    """
    out[i·s + a, j·s + b] += y[i, j] · w[a, b]      for every input (i,j), kernel offset (a,b)
    out size = (H−1)·s + k
    """
    H, W = y.shape
    k = w.shape[0]
    out = torch.zeros((H - 1) * stride + k, (W - 1) * stride + k)
    for i in range(H):
        for j in range(W):
            out[i * stride : i * stride + k, j * stride : j * stride + k] += y[i, j] * w
    return out

In [5]:
y = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
w = torch.tensor([[1.0, 0.0, 0.0], [0.0, 10.0, 0.0], [0.0, 0.0, 100.0]])
convT.weight.data = w.reshape(1, 1, 3, 3)
mine = conv_transpose_by_hand(y, w)
torch_out = convT(y.reshape(1, 1, 2, 2))[0, 0]
mine, (mine - torch_out).abs().max()

(tensor([[  1.,   2.,   0.,   0.],
         [  3.,  14.,  20.,   0.],
         [  0.,  30., 140., 200.],
         [  0.,   0., 300., 400.]]),
 tensor(0., grad_fn=<MaxBackward1>))

Each input pixel leaves a diagonal stamp (1, 10, 100 × its value); overlaps add. Max diff 0.


## 4. Why "transpose": as matrices

Conv is linear, so `out_flat = C @ x_flat` for some matrix C (4 × 16 here). Build C by feeding basis images. ConvTranspose2d with the **same weights** is exactly `C.T` (16 × 4).


In [6]:
conv.weight.data = torch.randn(1, 1, 3, 3)
basis = torch.eye(16).reshape(16, 1, 4, 4)
Cmat = conv(basis).reshape(16, 4).T  # column n = conv of basis image n
Cmat.shape

torch.Size([4, 16])

In [7]:
convT.weight.data = conv.weight.data.clone()  # same kernel, not flipped
y = torch.randn(1, 1, 2, 2)
via_matrix = (Cmat.T @ y.flatten()).reshape(4, 4)
(via_matrix - convT(y)[0, 0]).abs().max()

tensor(0., grad_fn=<MaxBackward1>)

Diff ~0. So ConvTranspose2d is **not the inverse** of Conv2d (it won't give back x) — it's the transpose: it maps the conv's output _shape_ back to its input _shape_. Autograd uses exactly this: the gradient of a Conv2d w.r.t. its input is a ConvTranspose2d of the upstream grad.


In [8]:
x = torch.randn(1, 1, 4, 4, requires_grad=True)
g = torch.randn(1, 1, 2, 2)
conv(x).backward(g)
(x.grad - convT(g)).abs().max()

tensor(0., grad_fn=<MaxBackward1>)

## 5. Stride 2: the decoder case

Conv2d k=2 s=2 halves; ConvTranspose2d k=2 s=2 doubles, each input pixel stamps its own 2×2 block, no overlap.


In [9]:
down = nn.Conv2d(1, 1, kernel_size=2, stride=2, bias=False)
up = nn.ConvTranspose2d(1, 1, kernel_size=2, stride=2, bias=False)
nn.init.ones_(up.weight)
x = torch.randn(1, 1, 8, 8)
y = torch.tensor([[[[1.0, 2.0], [3.0, 4.0]]]])
down(x).shape, up(y)

(torch.Size([1, 1, 4, 4]),
 tensor([[[[1., 1., 2., 2.],
           [1., 1., 2., 2.],
           [3., 3., 4., 4.],
           [3., 3., 4., 4.]]]], grad_fn=<ConvolutionBackward0>))

## 6. Shape formulas

```
Conv2d:          H_out = ⌊(H + 2p − k) / s⌋ + 1
ConvTranspose2d: H_out = (H − 1)·s − 2p + k + output_padding
```

The second is the first solved for H: the input size a Conv2d would need to produce H.
The ⌊⌋ means several H map to the same H_out (e.g. s=2: H=7 and H=8 both → 4), so ConvT can't know which; `output_padding` picks.


In [10]:
for H in [7, 8]:
    print(H, '→', nn.Conv2d(1, 1, 3, stride=2, padding=1)(torch.randn(1, 1, H, H)).shape[-1])
t = nn.ConvTranspose2d(1, 1, 3, stride=2, padding=1)
z = torch.randn(1, 1, 4, 4)
t(z).shape[-1], nn.ConvTranspose2d(1, 1, 3, stride=2, padding=1, output_padding=1)(z).shape[-1]

7 → 4
8 → 4


(7, 8)

## 7. Same thing, as a normal conv

Transposed conv (stride s) = insert s−1 zeros between input pixels, pad by k−1, run an ordinary conv with the kernel **flipped**.


In [11]:
s, k = 2, 3
y = torch.randn(1, 1, 3, 3)
w = torch.randn(1, 1, k, k)
up_zeros = torch.zeros(1, 1, (3 - 1) * s + 1, (3 - 1) * s + 1)
up_zeros[..., ::s, ::s] = y  # zero-insert
as_conv = F.conv2d(F.pad(up_zeros, [k - 1] * 4), w.flip(-1, -2))
as_convT = F.conv_transpose2d(y, w, stride=s)
as_conv.shape, (as_conv - as_convT).abs().max()

(torch.Size([1, 1, 7, 7]), tensor(0.0))

## 8. The catch: checkerboard

k=3, s=2: stamps overlap on some cells and not others, so even an all-ones input with all-ones kernel comes out uneven. Pattern = stamp count.


In [12]:
y = torch.ones(1, 1, 4, 4)
w = torch.ones(1, 1, 3, 3)
F.conv_transpose2d(y, w, stride=2)[0, 0]

tensor([[1., 1., 2., 1., 2., 1., 2., 1., 1.],
        [1., 1., 2., 1., 2., 1., 2., 1., 1.],
        [2., 2., 4., 2., 4., 2., 4., 2., 2.],
        [1., 1., 2., 1., 2., 1., 2., 1., 1.],
        [2., 2., 4., 2., 4., 2., 4., 2., 2.],
        [1., 1., 2., 1., 2., 1., 2., 1., 1.],
        [2., 2., 4., 2., 4., 2., 4., 2., 2.],
        [1., 1., 2., 1., 2., 1., 2., 1., 1.],
        [1., 1., 2., 1., 2., 1., 2., 1., 1.]])

1s, 2s and 4s in a grid: a trained decoder has to fight this, and often leaves faint checkerboard artifacts. Fix: k divisible by s (k=4, s=2 → even overlap), or `Upsample(nearest) + Conv2d`.


In [13]:
F.conv_transpose2d(y, torch.ones(1, 1, 4, 4), stride=2, padding=1)[0, 0]

tensor([[1., 2., 2., 2., 2., 2., 2., 1.],
        [2., 4., 4., 4., 4., 4., 4., 2.],
        [2., 4., 4., 4., 4., 4., 4., 2.],
        [2., 4., 4., 4., 4., 4., 4., 2.],
        [2., 4., 4., 4., 4., 4., 4., 2.],
        [2., 4., 4., 4., 4., 4., 4., 2.],
        [2., 4., 4., 4., 4., 4., 4., 2.],
        [1., 2., 2., 2., 2., 2., 2., 1.]])

k=4, s=2, padding=1: interior all 4s, even. Only the border is lower (the image edge, not a pattern).


## Buys / fails

- **Buys**: a learned upsampler; exact shape inverse of a Conv2d; it _is_ Conv2d's backward pass.
- **Fails**: not an inverse of the values; uneven overlap (k not divisible by s) → checkerboard; output size ambiguous for stride > 1 (`output_padding`).
